# W05-02 · 순위 결합과 재정렬

필수 20분 + 확장 15분. RRF의 동작을 수작업으로 확인하고 후보 생성과 재정렬을 분리합니다.

**진행 방식**: 먼저 실행 결과를 예측하고 셀을 실행합니다. 변경 실습은 제공된 값 하나를 바꿉니다. 핵심 셀은 네트워크·API 키 없이 실행됩니다. 모든 정책·질의는 교육용 합성 데이터입니다.

VS Code → 우측 상단 Select Kernel → 이 저장소의 .venv (Python 3.11). 노트북별로 Restart 후 Run All 가능합니다.

In [ ]:
from pathlib import Path
import sys, json, math, statistics
from dataclasses import asdict, replace
from collections import Counter
from time import perf_counter

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course_labs").is_dir()), None)
assert ROOT is not None, "저장소 내부에서 notebook을 열어 주세요."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_labs.retrieval import (
    Document,
    Chunk,
    LexicalIndex,
    load_documents,
    load_cases,
    chunk_document,
    tokenize,
    expand_query,
    eligible_chunks,
    reciprocal_rank_fusion,
    rerank_by_coverage,
    retrieval_metrics,
    retrieve,
    answer_from_evidence,
    evaluate,
)
from IPython.display import display, HTML

docs = load_documents()
cases = load_cases()
print("Python:", sys.version.split()[0], "| 문서:", len(docs), "| 평가 질의:", len(cases))
assert sys.version_info[:2] == (3, 11), "과정 기준 Python 3.11.x 커널 선택"

In [ ]:
def show(rows, columns=None):
    import html

    if not rows:
        return display(HTML("<p>결과 없음</p>"))
    cols = columns or list(rows[0])
    headers = "".join(
        '<th style="padding:8px;text-align:left">' + html.escape(str(c)) + "</th>" for c in cols
    )
    body = "".join(
        "<tr>"
        + "".join(
            '<td style="padding:8px;border-bottom:1px solid #ddd">'
            + html.escape(str(r.get(c, "")))
            + "</td>"
            for c in cols
        )
        + "</tr>"
        for r in rows
    )
    display(
        HTML(
            '<table style="font-size:15px;border-collapse:collapse"><tr>'
            + headers
            + "</tr>"
            + body
            + "</table>"
        )
    )

## 01 후보 목록

각 검색기가 서로 다른 순위를 낼 수 있습니다.

In [ ]:
q = "상품 반품 기간"
chunks = eligible_chunks([c for d in docs for c in chunk_document(d)], "2026-09-29")
index = LexicalIndex(chunks)
bm = index.search(q, "bm25", 8)
tf = index.search(q, "tfidf", 8)
show(
    [
        {
            "rank": i + 1,
            "bm25": bm[i]["chunk"]["id"] if i < len(bm) else "",
            "tfidf": tf[i]["chunk"]["id"] if i < len(tf) else "",
        }
        for i in range(max(len(bm), len(tf)))
    ]
)

**관찰·변경 과제**

한 검색기에만 포함된 후보도 결합 결과에 남는지 예측하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 02 RRF 수작업

Reciprocal Rank Fusion: 순위 역수를 합하는 방식입니다.

In [ ]:
constant = 60
print("두 검색기 1위+3위:", 1 / (constant + 1) + 1 / (constant + 3))
print("한 검색기 1위만:", 1 / (constant + 1))

**관찰·변경 과제**

constant=1과 60에서 상위 순위 강조 정도가 어떻게 달라지는지 비교하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 03 실제 결합

서로 다른 척도의 raw score 대신 순위로 결합합니다.

In [ ]:
fused = reciprocal_rank_fusion([bm, tf], constant=60, k=8)
show(
    [
        {"rank": i + 1, "chunk": h["chunk"]["id"], "rrf_score": round(h["score"], 6)}
        for i, h in enumerate(fused)
    ]
)

**관찰·변경 과제**

BM25 1개와 TF-IDF 1개를 결합한 실습이며 dense+sparse 운영 하이브리드와 구분하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 04 중복과 동점

같은 검색 목록의 중복 후보는 한 번만 기여합니다.

In [ ]:
h = {"chunk": {"id": "same"}, "score": 99}
score = reciprocal_rank_fusion([[h, h], [h]], 60)[0]["score"]
assert math.isclose(score, 2 / 61)
print(score)

**관찰·변경 과제**

동점의 정렬 규칙을 명시하지 않으면 재현이 왜 어려워지나요?

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 05 후보 수

재정렬기는 후보에 없는 정답을 복구할 수 없습니다.

In [ ]:
for candidates in [1, 3, 8]:
    hits = index.search("반품 배송비와 환불 처리 기간", "bm25", candidates)
    print(candidates, [h["chunk"]["doc_id"] for h in hits])

**관찰·변경 과제**

정답 문서가 후보에 없는 실패는 재정렬보다 어느 앞 단계에서 해결해야 할까요?

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 06 규칙 재정렬

여기서는 질문 단어의 원문 포함 비율로 순서를 바꿉니다.

In [ ]:
q = "반품 배송비와 환불 처리 기간"
hits = retrieve(q, method="rrf", k=5)["hits"]
reranked = rerank_by_coverage(q, hits)
show(
    [
        {"doc": h["chunk"]["doc_id"], "coverage": h["score"], "original_rank": h["original_rank"]}
        for h in reranked
    ]
)

**관찰·변경 과제**

학습된 cross-encoder가 아니라는 한계와 실제 reranker 대체 시 검증 항목을 쓰세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 07 복합 질문 분리

질문을 나눠 여러 근거를 검색하는 대안을 비교합니다.

In [ ]:
subqueries = ["반품 배송비", "환불 처리 기간"]
for sub in subqueries:
    print(sub, [h["chunk"]["doc_id"] for h in retrieve(sub, k=2)["hits"]])

**관찰·변경 과제**

추가 검색 횟수·지연·잘못된 질문 분해를 함께 고려하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 08 점수 정규화 실험

단순 min-max는 후보 구성에 민감합니다.

In [ ]:
scores = [2, 3, 4]
normalized = [(s - min(scores)) / (max(scores) - min(scores)) for s in scores]
outlier = [2, 3, 4, 100]
print(normalized, [(s - min(outlier)) / (max(outlier) - min(outlier)) for s in outlier])

**관찰·변경 과제**

새 후보 하나로 기존 점수 비중이 달라지는 현상을 설명하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 09 전체 파이프라인

모듈별 변경과 실패 위치를 기록합니다.

In [ ]:
r = retrieve("상품 되돌려 보내는 기간", method="rrf", expand=True, rerank=True)
print(r["config"])
print([h["chunk"]["doc_id"] for h in r["hits"]])
print(answer_from_evidence(r)["text"])

**관찰·변경 과제**

확장·결합·재정렬을 한꺼번에 켜기 전에 각 효과를 따로 실험하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 10 변경 승인 기준

평균 개선만이 아니라 악화 사례와 비용도 판단합니다.

In [ ]:
experiments = [
    evaluate([c for c in cases if c["split"] == "dev"], method=m, k=3)
    for m in ["bm25", "tfidf", "rrf"]
]
show([{"method": e["config"]["method"], **e["summary"]} for e in experiments])

**관찰·변경 과제**

평균이 같을 때 운영 복잡도가 더 높은 방법을 선택할 근거가 있나요?

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 완료 체크

- 설정 하나를 바꾼 비교 결과와 실패 질의 1개
- 출처·문서 버전·기준 날짜를 포함한 설명
- 측정한 것과 측정하지 않은 것의 구분

## 참고

- [RAG 원 논문](https://arxiv.org/abs/2005.11401)
- [Stanford IR: TF-IDF](https://nlp.stanford.edu/IR-book/html/htmledition/term-frequency-and-weighting-1.html)
- [Stanford IR: BM25](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html)
- [Ollama embed API](https://docs.ollama.com/api/embed)

로컬 구현 위치: course_labs/retrieval.py. 형태소 분석·학습된 임베딩·신경망 재정렬을 구현한 것으로 해석하지 않습니다.